# Lab 3 — Detección de Anomalías en Tráfico de Red
**Curso:** Seguridad Informática | **Unidad IV:** Monitoreo, SIEM e IA

Modelo: Isolation Forest sobre `network_traffic.csv` (10 000 registros).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

sns.set_theme(style='whitegrid')
DATASET = 'network_traffic.csv'

## 3.1 — Exploración y preprocesamiento

In [ ]:
df = pd.read_csv(DATASET)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'Registros: {len(df)} | Columnas: {list(df.columns)}')
print(f'\nDistribución de labels:\n{df["label"].value_counts()}')
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['bytes_sent'], bins=50, color='#2563eb', edgecolor='white')
axes[0].set_title('Distribución de bytes_sent')
axes[0].set_xlabel('bytes_sent')
axes[1].hist(df['duration_sec'], bins=50, color='#dc2626', edgecolor='white')
axes[1].set_title('Distribución de duration_sec')
axes[1].set_xlabel('duration_sec')
plt.tight_layout()
plt.show()

In [ ]:
print('Valores nulos:')
print(df.isnull().sum())

# Tratar outliers extremos (percentil 99)
for col in ('bytes_sent', 'bytes_recv', 'duration_sec', 'packets'):
    limite = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=limite)
    print(f'{col}: clip en P99 = {limite:.2f}')

In [ ]:
# Feature engineering
df['ratio_bytes'] = df['bytes_sent'] / (df['bytes_recv'] + 1)
df['bytes_por_segundo'] = (df['bytes_sent'] + df['bytes_recv']) / (df['duration_sec'] + 0.001)
df['packets_por_segundo'] = df['packets'] / (df['duration_sec'] + 0.001)
df['log_bytes_sent'] = np.log1p(df['bytes_sent'])
df['log_bytes_recv'] = np.log1p(df['bytes_recv'])

le = LabelEncoder()
df['protocol_enc'] = le.fit_transform(df['protocol'])

feature_cols = [
    'dst_port', 'bytes_sent', 'bytes_recv', 'duration_sec', 'packets',
    'ratio_bytes', 'bytes_por_segundo', 'packets_por_segundo',
    'log_bytes_sent', 'log_bytes_recv', 'protocol_enc'
]

X = df[feature_cols].copy()
y = (df['label'] == 'anomaly').astype(int)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols, index=df.index)
X_scaled.head()

## 3.2 — Entrenamiento del modelo Isolation Forest

In [ ]:
model = IsolationForest(contamination=0.04, n_estimators=200, random_state=42)
model.fit(X_scaled)

pred = model.predict(X_scaled)
y_pred = (pred == -1).astype(int)

print('=== Métricas de evaluación ===')
print(f'Precision: {precision_score(y, y_pred):.4f}')
print(f'Recall:    {recall_score(y, y_pred):.4f}')
print(f'F1-Score:  {f1_score(y, y_pred):.4f}')
print('\nMatriz de confusión:')
cm = confusion_matrix(y, y_pred)
print(cm)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomalía'],
            yticklabels=['Normal', 'Anomalía'], ax=ax)
ax.set_title('Matriz de confusión — Isolation Forest')
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

## 3.3 — Interpretación y umbral dinámico

In [ ]:
scores = -model.decision_function(X_scaled)

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(scores[y == 0], bins=60, alpha=0.6, label='Normal', color='#2563eb')
ax.hist(scores[y == 1], bins=60, alpha=0.6, label='Anomalía', color='#dc2626')
ax.set_title('Distribución del score de anomalía (decision_function)')
ax.set_xlabel('Score (mayor = más anómalo)')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
umbrales = np.linspace(scores.min(), scores.max(), 300)
f1_scores = []
for umbral in umbrales:
    y_u = (scores >= umbral).astype(int)
    f1_scores.append(f1_score(y, y_u, zero_division=0))

idx_opt = int(np.argmax(f1_scores))
umbral_optimo = umbrales[idx_opt]
f1_optimo = f1_scores[idx_opt]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(umbrales, f1_scores, color='#059669', linewidth=2)
ax.axvline(umbral_optimo, color='#dc2626', linestyle='--',
           label=f'Umbral óptimo={umbral_optimo:.4f} (F1={f1_optimo:.4f})')
ax.set_title('Curva Umbral vs F1-Score')
ax.set_xlabel('Umbral de score')
ax.set_ylabel('F1-Score')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Umbral óptimo: {umbral_optimo:.6f} | F1 máximo: {f1_optimo:.4f}')

In [ ]:
df['anomaly_score'] = scores
top10 = df.nlargest(10, 'anomaly_score')
top10

### Interpretación del Top 10 anómalo

Los registros más anómalos presentan patrones típicos de amenazas reales:

1. **Exfiltración de datos:** volúmenes extremos de `bytes_sent` con duraciones cortas (transferencia rápida de datos sensibles).
2. **Escaneo de puertos / reconocimiento:** alto `packets_por_segundo` con pocos bytes, indicando sondas automatizadas.
3. **Comunicación C2 sospechosa:** conexiones UDP/TCP a puertos inusuales con ratios bytes asimétricos.
4. **Beaconing:** conexiones periódicas de bajo volumen pero alta frecuencia de paquetes.
5. **Túneles de datos:** `bytes_por_segundo` desproporcionado respecto al tráfico normal de la red interna.

Estos patrones se desvían significativamente del comportamiento baseline aprendido por el Isolation Forest.

## 3.4 — Exportación del modelo

In [ ]:
artefacto = {
    'model': model,
    'scaler': scaler,
    'label_encoder': le,
    'feature_cols': feature_cols,
    'umbral_optimo': float(umbral_optimo),
}
joblib.dump(artefacto, 'modelo_anomalias.pkl')
print('Modelo exportado: modelo_anomalias.pkl')